# Lab 9: Dialogue Manager for Clarification Behaviour (Student Version)
## Follow-up, Explicit Clarification, and Implicit Clarification under OOV and Simulated Noisy ASR

In this lab, you will extend the restaurant booking agent from Lab 5 with a lightweight dialogue manager (DM).

You will **not** build live speech input. Instead, you will work with **simulated noisy ASR transcripts** and lightweight confidence-style matching scores, such as:
- `i want a cheap chainese place`
- `find me something in the senter`
- `book a place at seaside`

Your job is to implement the DM logic that decides:
1. whether a user turn needs a **follow-up**, **explicit clarification**, or **implicit clarification**
2. how confidence affects that decision
3. how the dialogue state should be updated safely

Complete each TODO cell, then run the demos and evaluation at the end.


In [ ]:
import json
import os
import re
import time
import random
import difflib
import requests
from typing import Any, Dict, List, Optional

Check if Ollama is Installed

In [ ]:
!ollama --version

ollama version is 0.18.1


Please verify that the model has been successfully deployed on your local machine. 

In [ ]:
!ollama list

NAME                           ID              SIZE      MODIFIED    
llama3.1:8b-instruct-q4_K_M    46e0c10c039e    4.9 GB    5 weeks ago    


Pull the model (skip if you already deployed the model)

In [ ]:
!ollama pull llama3.1:8b-instruct-q4_K_M

## Method Overview

We keep the original restaurant domain and tool-calling structure, but replace prompt-only clarification with a lightweight dialogue manager.

The dialogue manager:
1. tracks slot values in an explicit state
2. normalises noisy user input
3. estimates a lightweight confidence score from exact, substring, or fuzzy matches
4. chooses between **follow-up**, **explicit clarification**, **implicit clarification**, or **no clarification**

This allows us to evaluate clarification behaviour systematically.


### Teaching Notes

- **explicit clarification**: directly confirm or repair an uncertain slot value  
  Example: *"I think you said chinese. Is that right?"*
- **implicit clarification**: move to the next question while implicitly carrying forward a medium-confidence interpretation  
  Example: *"OK, in what area are you looking for a chinese restaurant?"*
- **follow-up**: ask for a slot that has not been specified yet  
  Example: *"What area are you looking for?"*

Your task is to implement that policy in code.


Generate the restaurant database

In [ ]:
restaurants = [
    {"id": "r001", "name": "Centre Noodle House", "area": "centre", "food": "chinese", "pricerange": "cheap", "address": "11 King St", "phone": "01223 100001"},
    {"id": "r002", "name": "Centre La Pasta", "area": "centre", "food": "italian", "pricerange": "moderate", "address": "4 Market Sq", "phone": "01223 100002"},
    {"id": "r003", "name": "Centre Spice Garden", "area": "centre", "food": "indian", "pricerange": "expensive", "address": "25 Regent St", "phone": "01223 100003"},
    {"id": "r004", "name": "Centre Sakura", "area": "centre", "food": "japanese", "pricerange": "moderate", "address": "8 Trinity Ln", "phone": "01223 100004"},
    {"id": "r005", "name": "West Green Bowl", "area": "west", "food": "vegetarian", "pricerange": "cheap", "address": "3 Willow Rd", "phone": "01223 100005"},
    {"id": "r006", "name": "West Dragon", "area": "west", "food": "chinese", "pricerange": "moderate", "address": "18 Queen Rd", "phone": "01223 100006"},
    {"id": "r007", "name": "North Grill", "area": "north", "food": "british", "pricerange": "moderate", "address": "77 Milton Rd", "phone": "01223 100007"},
    {"id": "r008", "name": "Riverside Curry House", "area": "riverside", "food": "indian", "pricerange": "cheap", "address": "2 River Walk", "phone": "01223 100008"},
]

with open("restaurants.json", "w", encoding="utf-8") as f:
    json.dump(restaurants, f, ensure_ascii=False, indent=2)

with open("restaurants.json", "r", encoding="utf-8") as f:
    RESTAURANTS = json.load(f)

print(f"Loaded {len(RESTAURANTS)} restaurants.")

Loaded 8 restaurants.


Basic tool functions

In [ ]:
BOOKINGS: Dict[str, Dict[str, Any]] = {}

def _norm(x: Optional[str]) -> Optional[str]:
    if x is None:
        return None
    x = str(x).strip().lower()
    return x if x else None

def search_restaurants(
    area: Optional[str] = None,
    food: Optional[str] = None,
    pricerange: Optional[str] = None,
    name: Optional[str] = None,
    limit: int = 5,
) -> Dict[str, Any]:
    area_n = _norm(area)
    food_n = _norm(food)
    price_n = _norm(pricerange)
    name_n = _norm(name)

    results = []
    for r in RESTAURANTS:
        if area_n and r["area"] != area_n:
            continue
        if food_n and r["food"] != food_n:
            continue
        if price_n and r["pricerange"] != price_n:
            continue
        if name_n and name_n not in r["name"].lower():
            continue
        results.append(r)

    limit = max(1, min(int(limit), 10))
    return {"ok": True, "count": len(results), "results": results[:limit]}

def book_restaurant(
    restaurant_id: str,
    day: str,
    time_str: str,
    people: int,
    customer_name: str,
) -> Dict[str, Any]:
    known_ids = {r["id"] for r in RESTAURANTS}
    if restaurant_id not in known_ids:
        return {"ok": False, "error": f"Unknown restaurant_id: {restaurant_id}"}

    try:
        people = int(people)
    except Exception:
        return {"ok": False, "error": "people must be an integer"}

    if people < 1 or people > 20:
        return {"ok": False, "error": "people must be between 1 and 20"}

    if not str(day).strip():
        return {"ok": False, "error": "day is required"}
    if not str(time_str).strip():
        return {"ok": False, "error": "time is required"}
    if not str(customer_name).strip():
        return {"ok": False, "error": "customer_name is required"}

    booking_id = f"BKG-{random.randint(100000, 999999)}"
    BOOKINGS[booking_id] = {
        "booking_id": booking_id,
        "restaurant_id": restaurant_id,
        "day": str(day).strip(),
        "time": str(time_str).strip(),
        "people": people,
        "customer_name": str(customer_name).strip(),
    }
    return {"ok": True, "booking": BOOKINGS[booking_id]}

def cancel_booking(booking_id: str) -> Dict[str, Any]:
    if booking_id not in BOOKINGS:
        return {"ok": False, "error": "unknown_booking_id"}
    removed = BOOKINGS.pop(booking_id)
    return {"ok": True, "cancelled": removed}


## Slot Ontology

The restaurant search task uses three main search constraints:
- area
- food
- pricerange

The booking stage additionally uses:
- restaurant_id
- day
- time
- number of guests
- customer_name

Slot maps

In [ ]:
AREA_MAP = {
    "centre": "centre",
    "center": "centre",
    "city centre": "centre",
    "city center": "centre",
    "downtown": "centre",
    "central": "centre",
    "north": "north",
    "south": "south",
    "east": "east",
    "west": "west",
    "riverside": "riverside",
    "river side": "riverside",
    "riverside area": "riverside",
}

FOOD_MAP = {
    "chinese": "chinese",
    "italian": "italian",
    "indian": "indian",
    "japanese": "japanese",
    "vegetarian": "vegetarian",
    "veggie": "vegetarian",
    "british": "british",
    "english": "british",
}

PRICE_MAP = {
    "cheap": "cheap",
    "inexpensive": "cheap",
    "budget": "cheap",
    "low": "cheap",
    "low priced": "cheap",
    "moderate": "moderate",
    "medium": "moderate",
    "moderately priced": "moderate",
    "expensive": "expensive",
    "pricey": "expensive",
    "high": "expensive",
    "high end": "expensive",
}

VALID_AREAS = ["centre", "north", "south", "east", "west", "riverside"]
VALID_FOODS = ["chinese", "italian", "indian", "japanese", "vegetarian", "british"]
VALID_PRICES = ["cheap", "moderate", "expensive"]


## Task 1: Slot normalisation

Complete the function below.

Goal:
- turn noisy text into a structured slot hypothesis
- return a confidence-style score
- keep `unknown` separate from `missing`


In [ ]:
def normalize_slot_verbose(value, mapping, fuzzy_cutoff=0.72):
    """
    Return a dictionary with the following fields:
    - raw: original input value
    - value: canonical mapped value or None
    - status: one of {"missing", "mapped", "substring", "fuzzy", "unknown"}
    - confidence: float confidence score
    - candidates: list of candidate canonical values
    """

    raw_value = value

    # 1) Missing
    if value is None:
        return {
            "raw": raw_value,
            "value": None,
            "status": "missing",
            "confidence": 0.0,
            "candidates": []
        }

    text = str(value).strip().lower()
    if not text:
        return {
            "raw": raw_value,
            "value": None,
            "status": "missing",
            "confidence": 0.0,
            "candidates": []
        }

    # Canonical values
    canonical_values = sorted(set(mapping.values()))

    # 2) Exact match on mapping key
    if text in mapping:
        return {
            "raw": raw_value,
            "value": mapping[text],
            "status": "mapped",
            "confidence": 1.0,
            "candidates": [mapping[text]]
        }

    # 3) Substring match on mapping key
    substring_hits = []
    for key, canonical in mapping.items():
        if key in text:
            substring_hits.append((key, canonical))

    if substring_hits:
        substring_hits.sort(key=lambda x: len(x[0]), reverse=True)
        best_key, best_value = substring_hits[0]

        candidate_values = []
        for _, canonical in substring_hits:
            if canonical not in candidate_values:
                candidate_values.append(canonical)

        return {
            "raw": raw_value,
            "value": best_value,
            "status": "substring",
            "confidence": 0.9,
            "candidates": candidate_values
        }

    # 4) Fuzzy matching over short token spans / n-grams
    tokens = re.findall(r"\w+", text)

    spans = set()
    if tokens:
        max_n = min(3, len(tokens))
        for n in range(1, max_n + 1):
            for i in range(len(tokens) - n + 1):
                spans.add(" ".join(tokens[i:i+n]))

    # Also compare the full text
    spans.add(text)

    best_score = 0.0
    best_key = None
    best_value = None

    for span in spans:
        for key, canonical in mapping.items():
            score = difflib.SequenceMatcher(None, span, key).ratio()
            if score > best_score:
                best_score = score
                best_key = key
                best_value = canonical

    if best_key is not None and best_score >= fuzzy_cutoff:
        fuzzy_candidates = []
        for key, canonical in mapping.items():
            key_score = difflib.SequenceMatcher(None, text, key).ratio()

            token_scores = []
            for span in spans:
                token_scores.append(difflib.SequenceMatcher(None, span, key).ratio())

            final_score = max([key_score] + token_scores) if token_scores else key_score

            if final_score >= fuzzy_cutoff and canonical not in fuzzy_candidates:
                fuzzy_candidates.append(canonical)

        return {
            "raw": raw_value,
            "value": best_value,
            "status": "fuzzy",
            "confidence": round(best_score, 3),
            "candidates": fuzzy_candidates if fuzzy_candidates else [best_value]
        }

    # 5) Unknown
    return {
        "raw": raw_value,
        "value": None,
        "status": "unknown",
        "confidence": 0.0,
        "candidates": canonical_values
    }

In [ ]:
# After you finish Task 1, run this cell to sanity-check your function.
# Example checks:
normalize_slot_verbose("chainese", FOOD_MAP)



{'raw': 'chainese',
 'value': 'chinese',
 'status': 'fuzzy',
 'confidence': 0.933,
 'candidates': ['chinese']}

User input parser with lightweight slot mention detection


In [ ]:
AREA_HINTS = [
    "area", "location", "part of town", "part", "side", "district",
    "centre", "center", "central", "north", "south", "east", "west",
    "riverside", "river side", "downtown"
]

FOOD_HINTS = [
    "food", "cuisine", "eat", "meal",
    "chinese", "italian", "indian", "japanese", "vegetarian", "veggie",
    "british", "english", "asian", "oriental"
]

PRICE_HINTS = [
    "price", "priced", "cost", "cheap", "budget", "expensive", "pricey",
    "moderate", "mid", "medium", "midscale", "premium", "high end"
]

def detect_slot_mention(text: str, hints: List[str]) -> bool:
    return any(h in text for h in hints)

def parse_slot_with_hints(text: str, mapping: Dict[str, str], hints: List[str]):
    info = normalize_slot_verbose(text, mapping)

    if info["status"] == "unknown":
        if detect_slot_mention(text, hints):
            info["mentioned"] = True
        else:
            info["status"] = "missing"
            info["mentioned"] = False
    else:
        info["mentioned"] = info["status"] != "missing"

    return info

def parse_user_constraints(user_text: str):
    text = user_text.lower()

    area_info = parse_slot_with_hints(text, AREA_MAP, AREA_HINTS)
    food_info = parse_slot_with_hints(text, FOOD_MAP, FOOD_HINTS)
    price_info = parse_slot_with_hints(text, PRICE_MAP, PRICE_HINTS)

    return {
        "area": area_info,
        "food": food_info,
        "pricerange": price_info,
    }


Booking field parser

In [ ]:
def extract_booking_fields(user_text: str):
    text = user_text.lower()

    people = None
    m_people = re.search(r"\bfor (\d+)\b", text) or re.search(r"\b(\d+) people\b", text)
    if m_people:
        people = int(m_people.group(1))

    time_str = None
    m_time = re.search(r"\b(\d{1,2}:\d{2})\b", text)
    if m_time:
        time_str = m_time.group(1)

    if not time_str:
        m_time2 = re.search(r"\bat (\d{1,2})\b", text)
        if m_time2:
            hh = int(m_time2.group(1))
            time_str = f"{hh:02d}:00"

    day = None
    for d in ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday", "today", "tomorrow"]:
        if d in text:
            day = d
            break

    return {
        "day": day,
        "time": time_str,
        "people": people,
    }

Dialogue state

In [ ]:
def init_state():
    return {
        "constraints": {
            "area": None,
            "food": None,
            "pricerange": None,
            "name": None,
        },
        "booking": {
            "restaurant_id": None,
            "day": None,
            "time": None,
            "people": None,
            "customer_name": None,
        },
        "pending_clarification": None,
        "last_results": [],
    }

STATE = init_state()
STATE

{'constraints': {'area': None, 'food': None, 'pricerange': None, 'name': None},
 'booking': {'restaurant_id': None,
  'day': None,
  'time': None,
  'people': None,
  'customer_name': None},
 'pending_clarification': None,
 'last_results': []}

## Task 2: Clarification policy

Complete the function below using lecture distinction.

Suggested policy:
- slot not specified -> follow-up
- slot specified, value vague/OOV -> explicit clarification
- slot specified, value given but low confidence -> explicit clarification
- slot specified, value given but medium confidence -> implicit clarification
- slot specified, value given with high confidence -> no clarification


In [ ]:
def get_next_missing_slot(state, exclude_slot=None):
    for slot in ["area", "food", "pricerange"]:
        if slot == exclude_slot:
            continue
        if state["constraints"][slot] is None:
            return slot
    return None


def decide_clarification(state, parsed_slots, low_conf=0.84, medium_conf=0.94):
    """
    Priority order:
    1) unknown/OOV -> explicit clarification
    2) low confidence -> explicit clarification
    3) medium confidence -> implicit clarification
    4) missing slot -> follow_up
    5) otherwise -> none
    """

    search_slots = ["area", "food", "pricerange"]

    # 1) Unknown / OOV
    for slot in search_slots:
        info = parsed_slots[slot]
        mentioned = info.get("mentioned", False)
        status = info.get("status")
        value = info.get("value")
        candidates = info.get("candidates", [])

        if mentioned and (status == "unknown" or (value is None and status != "missing")):
            return {
                "action": "explicit_clarification",
                "slot": slot,
                "value": None,
                "reason": "unknown_value",
                "candidates": candidates
            }

    # 2) Low confidence
    for slot in search_slots:
        info = parsed_slots[slot]
        mentioned = info.get("mentioned", False)
        status = info.get("status")
        value = info.get("value")
        confidence = info.get("confidence", 0.0)
        candidates = info.get("candidates", [])

        if not mentioned or value is None:
            continue

        if status in {"mapped", "substring"}:
            continue

        if confidence < low_conf:
            return {
                "action": "explicit_clarification",
                "slot": slot,
                "value": value,
                "reason": "low_confidence",
                "candidates": candidates
            }

    # 3) Medium confidence
    for slot in search_slots:
        info = parsed_slots[slot]
        mentioned = info.get("mentioned", False)
        status = info.get("status")
        value = info.get("value")
        confidence = info.get("confidence", 0.0)
        candidates = info.get("candidates", [])

        if not mentioned or value is None:
            continue

        if status in {"mapped", "substring"}:
            continue

        if low_conf <= confidence < medium_conf:
            return {
                "action": "implicit_clarification",
                "slot": slot,
                "value": value,
                "reason": "medium_confidence",
                "candidates": candidates
            }

    # 4) Missing slot
    for slot in search_slots:
        info = parsed_slots[slot]
        mentioned = info.get("mentioned", False)

        if not mentioned and state["constraints"].get(slot) is None:
            return {
                "action": "follow_up",
                "slot": slot,
                "value": None,
                "reason": "missing_slot",
                "candidates": []
            }

    # 5) None
    return {
        "action": "none",
        "slot": None,
        "value": None,
        "reason": "all_clear",
        "candidates": []
    }

## Task 3: Clarification / follow-up response builder

Complete the function below.

Minimum behaviours:
- follow_up -> ask for the missing slot
- explicit clarification -> directly confirm or repair the uncertain slot
- implicit clarification -> ask the next question while implicitly carrying forward the assumed value


In [ ]:
def slot_values_text(slot):
    if slot == "area":
        return "centre, north, south, east, west, or riverside"
    if slot == "food":
        return "chinese, italian, indian, japanese, vegetarian, or british"
    if slot == "pricerange":
        return "cheap, moderate, or expensive"
    return ""


def follow_up_question(slot):
    if slot == "area":
        return "What area are you looking for?"
    if slot == "food":
        return "What kind of food would you like?"
    if slot == "pricerange":
        return "What price range would you like?"
    return "Could you tell me a bit more?"


def build_clarification_response(clf):
    """
    Build a JSON-style response object for the policy.
    """

    action = clf.get("action")
    slot = clf.get("slot")
    value = clf.get("value")
    reason = clf.get("reason")
    candidates = clf.get("candidates", [])

    if action == "follow_up":
        response = follow_up_question(slot)
        return {
            "action": "follow_up",
            "type": "follow_up",
            "slot": slot,
            "response": response,
            "text": response
        }

    if action == "explicit_clarification":
        if reason == "unknown_value":
            options_text = slot_values_text(slot)
            response = f"Could you confirm the {slot}? Possible values are {options_text}."
        elif value is not None:
            response = f"I think you said {value} for {slot}. Is that right?"
        else:
            options_text = slot_values_text(slot)
            response = f"Could you confirm the {slot}? Possible values are {options_text}."

        return {
            "action": "explicit_clarification",
            "type": "explicit_clarification",
            "slot": slot,
            "value": value,
            "candidates": candidates,
            "response": response,
            "text": response
        }

    if action == "implicit_clarification":
        if slot == "area":
            response = f"OK, in what area are you looking for?"
        elif slot == "food":
            response = f"OK, what kind of food would you like?"
        elif slot == "pricerange":
            response = f"OK, what price range would you like?"
        else:
            response = "OK, could you tell me a bit more?"

        return {
            "action": "implicit_clarification",
            "type": "implicit_clarification",
            "slot": slot,
            "value": value,
            "response": response,
            "text": response
        }

    return {
        "action": "none",
        "type": "none",
        "slot": None,
        "response": "",
        "text": ""
    }

## Task 4: Safe state update

Only commit slot values when they are reliable enough.

Minimum requirement:
- write canonical values into state for high-confidence cases such as `mapped` and `substring`
- do not overwrite a good state value with `unknown`
- keep fuzzy values out of the state until they are confirmed

In [ ]:
def update_state_from_parsed(state, parsed_slots):
    """
    Safely update the dialogue state from parsed search slots.

    Rules:
    - write canonical values into state for high-confidence cases such as
      "mapped" and "substring"
    - do not overwrite a good state value with unknown
    - keep fuzzy values out of the state until they are confirmed
    """

    for slot in ["area", "food", "pricerange"]:
        info = parsed_slots.get(slot, {})

        status = info.get("status")
        value = info.get("value")
        mentioned = info.get("mentioned", False)

        # If the slot was not really mentioned, do nothing
        if not mentioned:
            continue

        # Safe commit only for high-confidence canonical cases
        if status in {"mapped", "substring"} and value is not None:
            state["constraints"][slot] = value
            continue

        # Do not overwrite good values with unknown
        if status in {"unknown", "missing", "fuzzy"}:
            continue

    return state

## Why still use Ollama?

The dialogue manager decides when clarification is needed, but Ollama is still used for:
- generating natural task-oriented responses
- summarising search results
- producing booking confirmations

So the system becomes DM-controlled rather than purely prompt-controlled.

Ollama config

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "llama3.1:8b-instruct-q4_K_M"

New system prompt

In [ ]:
SYSTEM_PROMPT = """
You are a task-oriented restaurant booking assistant.

You MUST output ONLY ONE valid JSON object per turn.

Allowed actions:
1) {"action":"search_restaurants","args":{"area":"centre","food":"chinese","pricerange":"cheap","name":null,"limit":5}}
2) {"action":"book_restaurant","args":{"restaurant_id":"r001","day":"friday","time":"19:00","people":2,"customer_name":"Alex"}}
3) {"action":"cancel_booking","args":{"booking_id":"BKG-123456"}}
4) {"action":"final","response":"English message"}

Rules:
- Never invent restaurants or booking ids.
- Use tool calls when searching or booking.
- If tool results contain restaurants, summarise the options clearly.
- If there are no results, ask the user to relax constraints.
- Before booking, make sure you know restaurant_id, day, time, people, and customer_name.
- Keep responses concise and polite.
"""

Ollama call function

In [ ]:
def call_ollama(messages, temperature=0.2, timeout=180):
    payload = {
        "model": MODEL,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature
        }
    }

    r = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    return data["message"]["content"]

Tool dispatcher

In [ ]:
def dispatch_tool(action: str, args: Dict[str, Any]):
    if action == "search_restaurants":
        return search_restaurants(**args)
    elif action == "book_restaurant":
        return book_restaurant(
            restaurant_id=args["restaurant_id"],
            day=args["day"],
            time_str=args["time"],
            people=args["people"],
            customer_name=args["customer_name"],
        )
    elif action == "cancel_booking":
        return cancel_booking(**args)
    else:
        return {"ok": False, "error": f"unknown_action: {action}"}

Allowed actions

In [ ]:
ALLOWED_ACTIONS = {
    "search_restaurants",
    "book_restaurant",
    "cancel_booking",
    "follow_up",
    "explicit_clarification",
    "implicit_clarification",
    "final",
}

## Task 5: DM preprocessing

Complete the function below.

Suggested steps:
1. parse slot constraints from user_text
2. update the dialogue state safely
3. decide whether a dialogue move is needed
4. if yes, return the response object and updated state
5. otherwise return None, updated state, and parsed slots

In [ ]:
def dm_preprocess(user_text: str, state: Dict[str, Any]):
    """
    1. parse slot constraints from user_text
    2. update the dialogue state safely
    3. decide whether a dialogue move is needed
    4. if yes, return the response object and updated state
    5. otherwise return None, updated state, and parsed slots
    """

    # Parse user constraints
    parsed_slots = parse_user_constraints(user_text)

    # update state safely
    state = update_state_from_parsed(state, parsed_slots)

    # decide whether a dialogue move is needed
    clf = decide_clarification(state, parsed_slots)

    # If a DM move is needed, build a response object now
    if clf.get("action") != "none":
        response_obj = build_clarification_response(clf)

        # Store pending clarification so the next user turn can be interpreted
        state["pending_clarification"] = clf

        return response_obj, state, parsed_slots

    state["pending_clarification"] = None
    return None, state, parsed_slots

Result summariser

In [ ]:
def format_search_results(tool_result):
    if not tool_result.get("ok"):
        return "Sorry, something went wrong during the search."

    if tool_result["count"] == 0:
        return "I couldn't find any matching restaurants. You could try changing the area, food type, or price range."

    lines = []
    for r in tool_result["results"]:
        lines.append(
            f'{r["name"]} ({r["id"]}) - {r["food"]}, {r["pricerange"]}, {r["area"]}, address: {r["address"]}'
        )
    joined = "\n".join(lines)
    return f"I found these options:\n{joined}\nPlease tell me which restaurant you would like to book."

 Main agent turn

In [ ]:
def agent_turn(messages, user_text, state, max_hops=5):
    trace = []

    # 1) DM first
    dm_obj, state, parsed = dm_preprocess(user_text, state)
    trace.append({
        "stage": "dm_preprocess",
        "parsed": parsed,
        "pending_clarification": state["pending_clarification"]
    })

    messages.append({"role": "user", "content": user_text})

    if dm_obj is not None:
        messages.append({"role": "assistant", "content": json.dumps(dm_obj, ensure_ascii=False)})
        return messages, dm_obj["response"], state, trace

    # 2) Otherwise continue with LLM + tools
    hops = 0
    final_reply = None

    while hops < max_hops:
        hops += 1

        raw = call_ollama(messages)
        trace.append({"stage": "ollama_raw", "hop": hops, "raw": raw})

        try:
            obj = json.loads(raw)
        except Exception:
            final_reply = "Sorry, I produced an invalid response."
            messages.append({"role": "assistant", "content": json.dumps({"action": "final", "response": final_reply})})
            break

        action = obj.get("action")
        if action not in ALLOWED_ACTIONS:
            final_reply = f"Sorry, unsupported action: {action}"
            messages.append({"role": "assistant", "content": json.dumps({"action": "final", "response": final_reply})})
            break

        if action == "final":
            final_reply = obj["response"]
            messages.append({"role": "assistant", "content": raw})
            break

        if action in {"search_restaurants", "book_restaurant", "cancel_booking"}:
            args = obj.get("args", {})
            tool_result = dispatch_tool(action, args)
            trace.append({"stage": "tool_result", "hop": hops, "action": action, "result": tool_result})

            if action == "search_restaurants" and tool_result.get("ok"):
                state["last_results"] = tool_result["results"]
                if tool_result["count"] > 0:
                    final_reply = format_search_results(tool_result)
                    messages.append({"role": "assistant", "content": json.dumps({"action": "final", "response": final_reply}, ensure_ascii=False)})
                    break
                else:
                    final_reply = "I couldn't find any matching restaurants. Please try relaxing one of your constraints."
                    messages.append({"role": "assistant", "content": json.dumps({"action": "final", "response": final_reply}, ensure_ascii=False)})
                    break

            if action == "book_restaurant":
                if tool_result.get("ok"):
                    booking = tool_result["booking"]
                    restaurant = next(r for r in RESTAURANTS if r["id"] == booking["restaurant_id"])
                    final_reply = (
                        f'Your booking is confirmed at {restaurant["name"]}.\n'
                        f'Address: {restaurant["address"]}\n'
                        f'Name: {booking["customer_name"]}\n'
                        f'Party size: {booking["people"]}\n'
                        f'Day: {booking["day"]}\n'
                        f'Time: {booking["time"]}\n'
                        f'Booking reference: {booking["booking_id"]}'
                    )
                else:
                    final_reply = f'Sorry, booking failed: {tool_result.get("error", "unknown error")}'
                messages.append({"role": "assistant", "content": json.dumps({"action": "final", "response": final_reply}, ensure_ascii=False)})
                break

            if action == "cancel_booking":
                if tool_result.get("ok"):
                    final_reply = f'Booking {tool_result["cancelled"]["booking_id"]} has been cancelled.'
                else:
                    final_reply = f'Sorry, cancellation failed: {tool_result.get("error", "unknown error")}'
                messages.append({"role": "assistant", "content": json.dumps({"action": "final", "response": final_reply}, ensure_ascii=False)})
                break

    if final_reply is None:
        final_reply = "Sorry, I could not complete the request."

    return messages, final_reply, state, trace

## Before Running the Demos

Make sure you have completed Tasks 1-5 first.

### What students should submit

1. Completed notebook code for all TODO cells
2. Result of Demo1,Demo2,Demo3,Demo4
3. Result of final evaluation

Initialise conversation

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
state = init_state()
BOOKINGS.clear()

print("System ready.")

System ready.


## Demo1：Normal case

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
state = init_state()
BOOKINGS.clear()
messages, reply, state, trace = agent_turn(
    messages,
    "I want a cheap Chinese restaurant in the centre.",
    state
)

print(reply)
trace

I found these options:
Centre Noodle House (r001) - chinese, cheap, centre, address: 11 King St
Please tell me which restaurant you would like to book.


[{'stage': 'dm_preprocess',
  'parsed': {'area': {'raw': 'i want a cheap chinese restaurant in the centre.',
    'value': 'centre',
    'status': 'substring',
    'confidence': 0.9,
    'candidates': ['centre'],
    'mentioned': True},
   'food': {'raw': 'i want a cheap chinese restaurant in the centre.',
    'value': 'chinese',
    'status': 'substring',
    'confidence': 0.9,
    'candidates': ['chinese'],
    'mentioned': True},
   'pricerange': {'raw': 'i want a cheap chinese restaurant in the centre.',
    'value': 'cheap',
    'status': 'substring',
    'confidence': 0.9,
    'candidates': ['cheap'],
    'mentioned': True}},
  'pending_clarification': None},
 {'stage': 'ollama_raw',
  'hop': 1,
  'raw': '{"action":"search_restaurants","args":{"area":"centre","food":"chinese","pricerange":"cheap","name":null,"limit":5}}'},
 {'stage': 'tool_result',
  'hop': 1,
  'action': 'search_restaurants',
  'result': {'ok': True,
   'count': 1,
   'results': [{'id': 'r001',
     'name': 'Cent

## Demo2：Missing slot

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
state = init_state()
BOOKINGS.clear()
messages, reply, state, trace = agent_turn(
    messages,
    "I want Chinese food.",
    state
)

print(reply)
trace

What area are you looking for?


[{'stage': 'dm_preprocess',
  'parsed': {'area': {'raw': 'i want chinese food.',
    'value': None,
    'status': 'missing',
    'confidence': 0.0,
    'candidates': ['centre', 'east', 'north', 'riverside', 'south', 'west'],
    'mentioned': False},
   'food': {'raw': 'i want chinese food.',
    'value': 'chinese',
    'status': 'substring',
    'confidence': 0.9,
    'candidates': ['chinese'],
    'mentioned': True},
   'pricerange': {'raw': 'i want chinese food.',
    'value': None,
    'status': 'missing',
    'confidence': 0.0,
    'candidates': ['cheap', 'expensive', 'moderate'],
    'mentioned': False}},
  'pending_clarification': {'action': 'follow_up',
   'slot': 'area',
   'value': None,
   'reason': 'missing_slot',
   'candidates': []}}]

## Demo3：OOV case

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
state = init_state()
BOOKINGS.clear()
messages, reply, state, trace = agent_turn(
    messages,
    "I want an oriental place downtown.",
    state
)

print(reply)
trace

Could you confirm the food? Possible values are chinese, italian, indian, japanese, vegetarian, or british.


[{'stage': 'dm_preprocess',
  'parsed': {'area': {'raw': 'i want an oriental place downtown.',
    'value': 'centre',
    'status': 'substring',
    'confidence': 0.9,
    'candidates': ['centre'],
    'mentioned': True},
   'food': {'raw': 'i want an oriental place downtown.',
    'value': None,
    'status': 'unknown',
    'confidence': 0.0,
    'candidates': ['british',
     'chinese',
     'indian',
     'italian',
     'japanese',
     'vegetarian'],
    'mentioned': True},
   'pricerange': {'raw': 'i want an oriental place downtown.',
    'value': None,
    'status': 'missing',
    'confidence': 0.0,
    'candidates': ['cheap', 'expensive', 'moderate'],
    'mentioned': False}},
  'pending_clarification': {'action': 'explicit_clarification',
   'slot': 'food',
   'value': None,
   'reason': 'unknown_value',
   'candidates': ['british',
    'chinese',
    'indian',
    'italian',
    'japanese',
    'vegetarian']}}]

## Demo4：Noisy ASR case

1. Missing words(in the)(Need implicit clarification or move on)

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
state = init_state()
BOOKINGS.clear()

messages, reply, state, trace = agent_turn(
    messages,
    "I want a cheap Chinese restaurant centre.",
    state
)

print(reply)
trace

I found these options:
Centre Noodle House (r001) - chinese, cheap, centre, address: 11 King St
Please tell me which restaurant you would like to book.


[{'stage': 'dm_preprocess',
  'parsed': {'area': {'raw': 'i want a cheap chinese restaurant centre.',
    'value': 'centre',
    'status': 'substring',
    'confidence': 0.9,
    'candidates': ['centre'],
    'mentioned': True},
   'food': {'raw': 'i want a cheap chinese restaurant centre.',
    'value': 'chinese',
    'status': 'substring',
    'confidence': 0.9,
    'candidates': ['chinese'],
    'mentioned': True},
   'pricerange': {'raw': 'i want a cheap chinese restaurant centre.',
    'value': 'cheap',
    'status': 'substring',
    'confidence': 0.9,
    'candidates': ['cheap'],
    'mentioned': True}},
  'pending_clarification': None},
 {'stage': 'ollama_raw',
  'hop': 1,
  'raw': '{"action":"search_restaurants","args":{"area":"centre","food":"chinese","pricerange":"cheap","name":null,"limit":5}}'},
 {'stage': 'tool_result',
  'hop': 1,
  'action': 'search_restaurants',
  'result': {'ok': True,
   'count': 1,
   'results': [{'id': 'r001',
     'name': 'Centre Noodle House',
   

2. Low confidence: Sounds like wrong slot(cent), need clarification

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
state = init_state()
BOOKINGS.clear()

messages, reply, state, trace = agent_turn(
    messages,
    "I want a cheap Chinese restaurant in the cent.",
    state
)

print(reply)
trace

I think you said centre for area. Is that right?


[{'stage': 'dm_preprocess',
  'parsed': {'area': {'raw': 'i want a cheap chinese restaurant in the cent.',
    'value': 'centre',
    'status': 'fuzzy',
    'confidence': 0.8,
    'candidates': ['centre'],
    'mentioned': True},
   'food': {'raw': 'i want a cheap chinese restaurant in the cent.',
    'value': 'chinese',
    'status': 'substring',
    'confidence': 0.9,
    'candidates': ['chinese'],
    'mentioned': True},
   'pricerange': {'raw': 'i want a cheap chinese restaurant in the cent.',
    'value': 'cheap',
    'status': 'substring',
    'confidence': 0.9,
    'candidates': ['cheap'],
    'mentioned': True}},
  'pending_clarification': {'action': 'explicit_clarification',
   'slot': 'area',
   'value': 'centre',
   'reason': 'low_confidence',
   'candidates': ['centre']}}]

3. Vague/OOV (No downtown option)(need explicit clarification)

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
state = init_state()
BOOKINGS.clear()

messages, reply, state, trace = agent_turn(
    messages,
    "I want a cheap Chinese restaurant in seaside.",
    state
)

print(reply)
trace

Could you confirm the area? Possible values are centre, north, south, east, west, or riverside.


[{'stage': 'dm_preprocess',
  'parsed': {'area': {'raw': 'i want a cheap chinese restaurant in seaside.',
    'value': None,
    'status': 'unknown',
    'confidence': 0.0,
    'candidates': ['centre', 'east', 'north', 'riverside', 'south', 'west'],
    'mentioned': True},
   'food': {'raw': 'i want a cheap chinese restaurant in seaside.',
    'value': 'chinese',
    'status': 'substring',
    'confidence': 0.9,
    'candidates': ['chinese'],
    'mentioned': True},
   'pricerange': {'raw': 'i want a cheap chinese restaurant in seaside.',
    'value': 'cheap',
    'status': 'substring',
    'confidence': 0.9,
    'candidates': ['cheap'],
    'mentioned': True}},
  'pending_clarification': {'action': 'explicit_clarification',
   'slot': 'area',
   'value': None,
   'reason': 'unknown_value',
   'candidates': ['centre', 'east', 'north', 'riverside', 'south', 'west']}}]

4. Slot not specified (need to follow-up)

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}]
state = init_state()
BOOKINGS.clear()

messages, reply, state, trace = agent_turn(
    messages,
    "I want a cheap restaurant.",
    state
)

print(reply)
trace

What area are you looking for?


[{'stage': 'dm_preprocess',
  'parsed': {'area': {'raw': 'i want a cheap restaurant.',
    'value': None,
    'status': 'missing',
    'confidence': 0.0,
    'candidates': ['centre', 'east', 'north', 'riverside', 'south', 'west'],
    'mentioned': False},
   'food': {'raw': 'i want a cheap restaurant.',
    'value': None,
    'status': 'missing',
    'confidence': 0.0,
    'candidates': ['british',
     'chinese',
     'indian',
     'italian',
     'japanese',
     'vegetarian'],
    'mentioned': False},
   'pricerange': {'raw': 'i want a cheap restaurant.',
    'value': 'cheap',
    'status': 'substring',
    'confidence': 0.9,
    'candidates': ['cheap'],
    'mentioned': True}},
  'pending_clarification': {'action': 'follow_up',
   'slot': 'area',
   'value': None,
   'reason': 'missing_slot',
   'candidates': []}}]

## Evaluation Setup

We evaluate the system on four groups of inputs:
1. follow-up / missing-slot cases
2. medium-confidence cases
3. OOV or vague-value cases
4. noisy ASR cases

For each case, we inspect:
- whether a dialogue move was triggered
- whether the move type was appropriate
- whether the response wording matches the intended policy


Evaluation sets

In [ ]:
TEST_MISSING = [
    {"id": "M1", "user": "I want Chinese food.", "expected_type": "follow_up"},
    {"id": "M2", "user": "I want a cheap place.", "expected_type": "follow_up"},
    {"id": "M3", "user": "Find me somewhere in the centre.", "expected_type": "follow_up"},
]

TEST_AMBIG = [
    {"id": "A1", "user": "I want chainese food.", "expected_type": "implicit"},
    {"id": "A2", "user": "Find me something in the senter.", "expected_type": "implicit"},
]

TEST_OOV = [
    {"id": "O1", "user": "I want asian food.", "expected_type": "explicit"},
    {"id": "O2", "user": "Find me a midscale place.", "expected_type": "explicit"},
    {"id": "O3", "user": "I'm looking for oriental food in the west.", "expected_type": "explicit"},
]

TEST_ASR = [
    {"id": "N1", "user": "i want a cheap chainese place", "expected_type": "implicit"},
    {"id": "N2", "user": "modrate japanees food", "expected_type": "implicit"},
    {"id": "N3", "user": "i want a restarant in the senter", "expected_type": "explicit"},
    {"id": "N4", "user": "book for to people at seven", "expected_type": "follow_up"},
]



Evaluation runner

In [ ]:
def classify_reply(reply_text):
    lower = reply_text.lower().strip()
    if lower.startswith("ok, in what area") or lower.startswith("ok, what kind of food") or lower.startswith("ok, what price range"):
        return "implicit"
    if lower.startswith("what area") or lower.startswith("what kind of food") or lower.startswith("what price range"):
        return "follow_up"
    if lower.startswith("i didn't catch") or lower.startswith("i think you said") or lower.startswith("could you confirm"):
        return "explicit"
    return "other"

def run_eval(testset, label):
    rows = []
    for item in testset:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        state = init_state()
        BOOKINGS.clear()

        messages, reply, state, trace = agent_turn(messages, item["user"], state)
        predicted = classify_reply(reply)

        rows.append({
            "group": label,
            "id": item["id"],
            "user": item["user"],
            "expected_type": item["expected_type"],
            "predicted_type": predicted,
            "match": predicted == item["expected_type"],
            "reply": reply
        })
    return rows


Run evaluation

In [ ]:
all_rows = []
all_rows += run_eval(TEST_MISSING, "missing")
all_rows += run_eval(TEST_AMBIG, "ambiguous")
all_rows += run_eval(TEST_OOV, "oov")
all_rows += run_eval(TEST_ASR, "noisy_asr")

for row in all_rows:
    print(row["group"], row["id"], row["match"], "->", row["reply"])

missing M1 True -> What area are you looking for?
missing M2 True -> What area are you looking for?
missing M3 True -> What kind of food would you like?
ambiguous A1 True -> OK, what kind of food would you like?
ambiguous A2 False -> I think you said centre for area. Is that right?
oov O1 True -> Could you confirm the food? Possible values are chinese, italian, indian, japanese, vegetarian, or british.
oov O2 True -> Could you confirm the pricerange? Possible values are cheap, moderate, or expensive.
oov O3 True -> Could you confirm the food? Possible values are chinese, italian, indian, japanese, vegetarian, or british.
noisy_asr N1 True -> OK, what kind of food would you like?
noisy_asr N2 True -> OK, what kind of food would you like?
noisy_asr N3 True -> I think you said centre for area. Is that right?
noisy_asr N4 True -> What area are you looking for?


Summary metric

In [ ]:
def simple_accuracy(rows):
    if not rows:
        return 0.0
    return sum(int(r["match"]) for r in rows) / len(rows)

print("Overall CR type accuracy:", simple_accuracy(all_rows))

Overall CR type accuracy: 0.9166666666666666
